In [1]:
# BiLSTM + TorchCRF (full notebook)
# Uses your original preprocessing and TorchCRF directly.
# Edit DATA_PATH at the top to point to your dataset, then run cells in order.


In [2]:
# !pip install -q torchcrf tqdm


In [3]:
import os
import unicodedata
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

try:
    from TorchCRF import CRF as TorchCRF_CRF
    print('Imported TorchCRF.CR F OK')
except Exception as e:
    TorchCRF_CRF = None
    print('Failed to import TorchCRF.CRF:', e)

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


d:\anaconda3\envs\pytorch118\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imported TorchCRF.CR F OK
PyTorch version: 2.5.1
CUDA available: True


In [4]:
# CONFIG - edit paths as needed
DATA_PATH = 'dataset/'
TRAIN_FILE = os.path.join(DATA_PATH, 'train.txt')
VAL_FILE = os.path.join(DATA_PATH, 'val.txt')
OUTPUT_MODEL_PATH = 'bilstm_crf_torchcrf_v2.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MAXLEN = 500
CHAR_EMB = 25
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 64
EPOCHS = 12
LR = 1e-3

PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'


In [5]:
def is_combining(ch):
    return unicodedata.category(ch) == 'Mn'

def split_char_diacritic_pairs(s):
    pairs = []
    base = None
    diacs = ''
    for ch in s:
        if is_combining(ch):
            if base is None:
                base = '<UNK_BASE>'
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base = ch
            diacs = ''
    if base is not None:
        pairs.append((base, diacs))
    return pairs

def is_arabic_letter(ch):
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    code = ord(ch)
    return (
        (0x0600 <= code <= 0x06FF) or
        (0x0750 <= code <= 0x077F) or
        (0x08A0 <= code <= 0x08FF) or
        (0xFB50 <= code <= 0xFDFF) or
        (0xFE70 <= code <= 0xFEFF)
    )

def placeholder_transform_pairs(pairs, placeholder=PLACEHOLDER):
    tokens = []
    labels = []
    for base, d in pairs:
        if isinstance(base, str) and base.startswith('<') and base.endswith('>'):
            tokens.append(base)
            labels.append('')
        elif base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append('')
        elif is_arabic_letter(base) or base == 'ـ':
            tokens.append(base)
            labels.append(d)
        else:
            tokens.append(placeholder)
            labels.append('')
    return tokens, labels


In [6]:
def load_lines_from_file(path):
    lines = []
    with open(path, 'r', encoding='utf8') as fh:
        for line in fh:
            s = line.strip()
            if s:
                lines.append(s)
    return lines

train_lines = load_lines_from_file(TRAIN_FILE) if os.path.exists(TRAIN_FILE) else []
val_lines = load_lines_from_file(VAL_FILE) if os.path.exists(VAL_FILE) else []
print('Train lines:', len(train_lines), 'Val lines:', len(val_lines))


Train lines: 50000 Val lines: 2500


In [7]:
train_tokens, train_labels = [], []
for line in train_lines:
    pairs = split_char_diacritic_pairs(line)
    toks, labs = placeholder_transform_pairs(pairs)
    train_tokens.append(toks)
    train_labels.append(labs)

val_tokens, val_labels = [], []
for line in val_lines:
    pairs = split_char_diacritic_pairs(line)
    toks, labs = placeholder_transform_pairs(pairs)
    val_tokens.append(toks)
    val_labels.append(labs)


In [8]:
char_counter = Counter(token for seq in (train_tokens + val_tokens) for token in seq)
special_chars = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN, SPACE_TOKEN, PLACEHOLDER]
single_chars = sorted([c for c in char_counter if len(c) == 1 and c not in special_chars])
multi_chars = sorted([c for c in char_counter if len(c) != 1 and c not in special_chars])
chars = special_chars + single_chars + multi_chars
char2idx = {c:i for i,c in enumerate(chars)}
idx2char = {i:c for c,i in char2idx.items()}

all_diacs = [d for lab in train_labels for d in lab]
from collections import Counter as _Counter
diag_counter = _Counter(all_diacs)
diac_classes = ['<PAD_LABEL>', '<NONE>']
most_common_diacs = [d for d,_ in diag_counter.most_common(13) if d != '']
for d in most_common_diacs:
    if d not in diac_classes:
        diac_classes.append(d)
diac_classes.append('<OTHER>')

diac2idx = {d:i for i,d in enumerate(diac_classes)}
idx2diac = {i:d for d,i in diac2idx.items()}
print('Vocab size:', len(char2idx), 'Diac labels:', len(diac2idx))


Vocab size: 45 Diac labels: 15


In [9]:
def map_diacritic_to_label(d):
    if d == '':
        return diac2idx['<NONE>']
    if d in diac2idx:
        return diac2idx[d]
    return diac2idx['<OTHER>']

def tokens_labels_to_ids(sequences_tokens, sequences_labels, maxlen=MAXLEN):
    X = []
    y = []
    for toks, labs in zip(sequences_tokens, sequences_labels):
        seq_mod = [SOS_TOKEN] + toks + [EOS_TOKEN]
        lab_mod = [''] + labs + ['']
        x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
        y_ids = [map_diacritic_to_label(d) for d in lab_mod]
        if len(x_ids) > maxlen:
            x_ids = x_ids[:maxlen]
            y_ids = y_ids[:maxlen]
        X.append(x_ids)
        y.append(y_ids)
    X_pad = []
    y_pad = []
    for seq_ids in X:
        if len(seq_ids) < maxlen:
            seq_ids = seq_ids + [char2idx[PAD_TOKEN]] * (maxlen - len(seq_ids))
        X_pad.append(seq_ids)
    for lab_ids in y:
        if len(lab_ids) < maxlen:
            lab_ids = lab_ids + [diac2idx['<PAD_LABEL>']] * (maxlen - len(lab_ids))
        y_pad.append(lab_ids)
    return np.array(X_pad, dtype=np.int64), np.array(y_pad, dtype=np.int64)

X_train_pad, y_train_pad = tokens_labels_to_ids(train_tokens, train_labels, MAXLEN)
X_val_pad, y_val_pad = tokens_labels_to_ids(val_tokens, val_labels, MAXLEN)
print('Shapes:', X_train_pad.shape if len(X_train_pad)>0 else (0,), y_train_pad.shape if len(y_train_pad)>0 else (0,), X_val_pad.shape if len(X_val_pad)>0 else (0,), y_val_pad.shape if len(y_val_pad)>0 else (0,))


Shapes: (50000, 500) (50000, 500) (2500, 500) (2500, 500)


In [10]:
class CharDiacDataset(Dataset):
    def __init__(self, X_arr, y_arr):
        self.X = X_arr
        self.y = y_arr
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

def collate_pad(batch, pad_idx=0, pad_label_idx=0):
    Xs, Ys = zip(*batch)
    lengths = [x.size(0) for x in Xs]
    maxlen = max(lengths)
    Xp = torch.full((len(Xs), maxlen), pad_idx, dtype=torch.long)
    Yp = torch.full((len(Xs), maxlen), pad_label_idx, dtype=torch.long)
    for i,(x,y) in enumerate(zip(Xs,Ys)):
        Xp[i,:x.size(0)] = x
        Yp[i,:y.size(0)] = y
    return Xp, Yp, torch.tensor(lengths, dtype=torch.long)

train_dataset = CharDiacDataset(X_train_pad, y_train_pad)
val_dataset = CharDiacDataset(X_val_pad, y_val_pad)
train_loader = DataLoader(train_dataset, batch_size=min(BATCH_SIZE, len(train_dataset)), shuffle=True, collate_fn=lambda b: collate_pad(b, char2idx[PAD_TOKEN], diac2idx['<PAD_LABEL>']))
val_loader = DataLoader(val_dataset, batch_size=min(BATCH_SIZE, len(val_dataset)), shuffle=False, collate_fn=lambda b: collate_pad(b, char2idx[PAD_TOKEN], diac2idx['<PAD_LABEL>']))
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))


Train batches: 782 Val batches: 40


In [11]:
class BiLSTM_Diac(nn.Module):
    def __init__(self, vocab_size, emb_dim, lstm_units, ff_units, num_labels, pad_idx=0, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.bilstm1 = nn.LSTM(emb_dim, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(2*lstm_units, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)
        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()
    def forward(self, x, lengths=None):
        emb = self.embedding(x)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out1, _ = self.bilstm1(packed)
            out1, _ = nn.utils.rnn.pad_packed_sequence(packed_out1, batch_first=True)
        else:
            out1, _ = self.bilstm1(emb)
        out1 = self.dropout1(out1)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(out1, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out2, _ = self.bilstm2(packed)
            out2, _ = nn.utils.rnn.pad_packed_sequence(packed_out2, batch_first=True)
        else:
            out2, _ = self.bilstm2(out1)
        out2 = self.dropout2(out2)
        ff = self.relu(self.ff1(out2))
        ff = self.relu(self.ff2(ff))
        logits = self.out(ff)
        return logits


In [12]:
if TorchCRF_CRF is None:
    raise RuntimeError('TorchCRF package not available. Install `TorchCRF` or adjust import.')

class BiLSTM_CRF_TorchCRF(nn.Module):
    def __init__(self, vocab_size, emb_dim, lstm_units, ff_units, num_labels, pad_idx=0, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.bilstm1 = nn.LSTM(emb_dim, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(2*lstm_units, lstm_units, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)
        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()
        # instantiate TorchCRF directly (no batch_first kw in constructor)
        self.crf = TorchCRF_CRF(num_labels, pad_idx=None, use_gpu=torch.cuda.is_available())

    def forward(self, x, lengths=None, tags=None, mask=None):
        emb = self.embedding(x)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out1, _ = self.bilstm1(packed)
            out1, _ = nn.utils.rnn.pad_packed_sequence(packed_out1, batch_first=True)
        else:
            out1, _ = self.bilstm1(emb)
        out1 = self.dropout1(out1)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(out1, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out2, _ = self.bilstm2(packed)
            out2, _ = nn.utils.rnn.pad_packed_sequence(packed_out2, batch_first=True)
        else:
            out2, _ = self.bilstm2(out1)
        out2 = self.dropout2(out2)
        ff = self.relu(self.ff1(out2))
        ff = self.relu(self.ff2(ff))
        emissions = self.out(ff)  # shape (B, T, num_labels)

        if mask is None:
            mask = (x != char2idx[PAD_TOKEN])

        if tags is not None:
            ll = self.crf(emissions, tags, mask)
            if isinstance(ll, torch.Tensor):
                loss = - ll.mean()
            else:
                loss = - torch.tensor(ll, dtype=emissions.dtype, device=emissions.device)
            return loss
        else:
            res = self.crf.viterbi_decode(emissions, mask)
            paths = [list(p) for p in res]
            return paths


In [13]:
vocab_size = len(char2idx)
num_labels = len(diac2idx)
pad_idx = char2idx[PAD_TOKEN]

model_bilstm = BiLSTM_Diac(vocab_size=vocab_size, emb_dim=CHAR_EMB, lstm_units=LSTM_UNITS, ff_units=FF_UNITS, num_labels=num_labels, pad_idx=pad_idx, dropout=DROPOUT).to(DEVICE)
model_crf = BiLSTM_CRF_TorchCRF(vocab_size=vocab_size, emb_dim=CHAR_EMB, lstm_units=LSTM_UNITS, ff_units=FF_UNITS, num_labels=num_labels, pad_idx=pad_idx, dropout=DROPOUT).to(DEVICE)

opt_bilstm = optim.Adam(model_bilstm.parameters(), lr=LR)
crit_bilstm = nn.CrossEntropyLoss(ignore_index=diac2idx['<PAD_LABEL>'])
opt_crf = optim.Adam(model_crf.parameters(), lr=LR)

print('Models instantiated on', DEVICE)


Models instantiated on cuda


In [14]:
def train_epoch_bilstm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    steps = 0
    for X, Y, lengths in tqdm(loader, desc='bilstm train'):
        X = X.to(device); Y = Y.to(device); lengths = lengths.to(device)
        optimizer.zero_grad()
        logits = model(X, lengths)
        b, T, C = logits.shape
        loss = criterion(logits.view(-1, C), Y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(1, steps)

def train_epoch_crf(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    steps = 0
    for X, Y, lengths in tqdm(loader, desc='crf train'):
        X = X.to(device); Y = Y.to(device); lengths = lengths.to(device)
        mask = (X != char2idx[PAD_TOKEN])
        optimizer.zero_grad()
        loss = model(X, lengths=lengths, tags=Y, mask=mask)
        if isinstance(loss, torch.Tensor) and loss.numel() > 1:
            loss = loss.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += float(loss.detach().cpu().item())
        steps += 1
    return total_loss / max(1, steps)

def predict_on_loader_collect_bilstm(model, loader, device):
    model.eval()
    Xs, Ys, Ypreds = [], [], []
    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch, lengths)
            pred_ids = logits.argmax(dim=-1).cpu().numpy()
            Xs.append(X_batch.cpu().numpy())
            Ys.append(y_batch.numpy())
            Ypreds.append(pred_ids)
    if len(Xs) == 0:
        return None, None, None
    return np.vstack(Xs), np.vstack(Ys), np.vstack(Ypreds)

def predict_on_loader_collect_crf(model, loader, device):
    model.eval()
    Xs, Ys, Ypreds = [], [], []
    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            X_batch = X_batch.to(device)
            mask = (X_batch != char2idx[PAD_TOKEN])
            preds = model(X_batch, lengths=lengths, tags=None, mask=mask)
            T = X_batch.size(1)
            batch_pred = np.full((len(preds), T), diac2idx['<PAD_LABEL>'], dtype=np.int64)
            for i, seq in enumerate(preds):
                L = min(len(seq), T)
                batch_pred[i, :L] = np.array(seq[:L], dtype=np.int64)
            Xs.append(X_batch.cpu().numpy())
            Ys.append(y_batch.numpy())
            Ypreds.append(batch_pred)
    if len(Xs) == 0:
        return None, None, None
    return np.vstack(Xs), np.vstack(Ys), np.vstack(Ypreds)

def compute_der_from_arrays(X_pad, y_true_pad, y_pred_pad, char2idx, diac2idx, idx2diac, exclude_case_ending=False):
    pad_label_id = diac2idx["<PAD_LABEL>"]
    pad_char_id = char2idx[PAD_TOKEN]
    sos_id = char2idx.get(SOS_TOKEN, None)
    eos_id = char2idx.get(EOS_TOKEN, None)
    space_id = char2idx.get(SPACE_TOKEN, None)
    placeholder_id = char2idx.get(PLACEHOLDER, None)

    N, T = X_pad.shape
    total = 0
    incorrect = 0

    for i in range(N):
        for t in range(T):
            ch_id = int(X_pad[i, t])
            if ch_id == pad_char_id:
                break
            if sos_id is not None and ch_id == sos_id:
                continue
            if eos_id is not None and ch_id == eos_id:
                continue

            gold = int(y_true_pad[i, t])
            if gold == pad_label_id:
                continue

            if exclude_case_ending:
                next_id = pad_char_id
                if t + 1 < T:
                    next_id = int(X_pad[i, t+1])
                is_word_final = False
                if next_id == pad_char_id:
                    is_word_final = True
                elif space_id is not None and next_id == space_id:
                    is_word_final = True
                elif placeholder_id is not None and next_id == placeholder_id:
                    is_word_final = True
                elif eos_id is not None and next_id == eos_id:
                    is_word_final = True

                if is_word_final:
                    continue

            pred = int(y_pred_pad[i, t])
            total += 1
            if pred != gold:
                incorrect += 1

    der = (incorrect / total) * 100.0 if total > 0 else None
    return der, incorrect, total


In [15]:
COMPARE_EPOCHS = 3
print('Starting comparison on device', DEVICE)
for ep in range(1, COMPARE_EPOCHS+1):
    print('--- Epoch {}/{} ---'.format(ep, COMPARE_EPOCHS))
    tr_b = train_epoch_bilstm(model_bilstm, train_loader, opt_bilstm, crit_bilstm, DEVICE)
    tr_c = train_epoch_crf(model_crf, train_loader, opt_crf, DEVICE)
    print(f'Train losses: BiLSTM={tr_b:.4f}  BiLSTM+CRF={tr_c:.4f}')

    Xb, yb, pb = predict_on_loader_collect_bilstm(model_bilstm, val_loader, DEVICE)
    Xc, yc, pc = predict_on_loader_collect_crf(model_crf, val_loader, DEVICE)
    if Xb is None or Xc is None:
        print('No validation predictions available')
        continue
    der_b_all, _, _ = compute_der_from_arrays(Xb, yb, pb, char2idx, diac2idx, idx2diac, exclude_case_ending=False)
    der_b_no, _, _ = compute_der_from_arrays(Xb, yb, pb, char2idx, diac2idx, idx2diac, exclude_case_ending=True)
    der_c_all, _, _ = compute_der_from_arrays(Xc, yc, pc, char2idx, diac2idx, idx2diac, exclude_case_ending=False)
    der_c_no, _, _ = compute_der_from_arrays(Xc, yc, pc, char2idx, diac2idx, idx2diac, exclude_case_ending=True)
    print(f'Validation DER (all):   BiLSTM={der_b_all:.4f}%   BiLSTM+CRF={der_c_all:.4f}%')
    print(f'Validation DER (noCE):  BiLSTM={der_b_no:.4f}%   BiLSTM+CRF={der_c_no:.4f}%')

print('Comparison finished')


Starting comparison on device cuda
--- Epoch 1/3 ---


crf train: 100%|██████████| 782/782 [37:15<00:00,  2.86s/it]    


Train losses: BiLSTM=0.4119  BiLSTM+CRF=74.7828
Validation DER (all):   BiLSTM=5.6044%   BiLSTM+CRF=4.8237%
Validation DER (noCE):  BiLSTM=4.8978%   BiLSTM+CRF=4.1955%
--- Epoch 2/3 ---


crf train: 100%|██████████| 782/782 [07:49<00:00,  1.67it/s]


Train losses: BiLSTM=0.1566  BiLSTM+CRF=6510.9033
Validation DER (all):   BiLSTM=3.8291%   BiLSTM+CRF=61.3549%
Validation DER (noCE):  BiLSTM=3.2889%   BiLSTM+CRF=62.0609%
--- Epoch 3/3 ---


crf train: 100%|██████████| 782/782 [08:01<00:00,  1.62it/s]


Train losses: BiLSTM=0.1236  BiLSTM+CRF=35669.0299
Validation DER (all):   BiLSTM=3.1630%   BiLSTM+CRF=89.0124%
Validation DER (noCE):  BiLSTM=2.6455%   BiLSTM+CRF=88.8107%
Comparison finished


In [16]:
torch.save({'model_state_dict': model_crf.state_dict(), 'char2idx': char2idx, 'diac2idx': diac2idx}, OUTPUT_MODEL_PATH)
print('Saved CRF checkpoint to', OUTPUT_MODEL_PATH)


Saved CRF checkpoint to bilstm_crf_torchcrf_v2.pt
